In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# The original project path was hard-coded to one computer. Run this notebook from the project root instead.
PROJECT_PATH = Path.cwd().resolve()

FEATURES_PATH = (
    PROJECT_PATH / "Dataset" / "Processed" / "Voice" /
    "Voice_Features" / "Voice_Combined_Features.csv"
)

MODEL_PATH = PROJECT_PATH / "Models" / "Voice"

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Voice Machine Learning Environment Configured.")
print(f"Features Path: {FEATURES_PATH}")
print(f"Model Path: {MODEL_PATH}")
print(f"Test Size: {TEST_SIZE}")

# 02 - Load Voice Features Dataset

This section loads the combined voice feature dataset from the CSV file.

The dataset contains MFCC, ZCR, and RMS features extracted from all audio recordings.

User_ID will be used as the target label, while Audio_Name will be retained only as metadata.

In [ ]:
# Load Combined Voice Features

voice_df = pd.read_csv(FEATURES_PATH)

print("Voice Features Dataset Loaded Successfully")
print("-" * 45)

print("Dataset Shape:", voice_df.shape)

print("\nUser Distribution:")
print(voice_df["User_ID"].value_counts())

print("\nMissing Values:", voice_df.isnull().sum().sum())

print("\nTotal Numerical Features:",
      voice_df.drop(columns=["User_ID", "Audio_Name"]).shape[1])

print("\nDataset Preview:")
display(voice_df.head())

# 03 - Separate Features and Target

This section separates the numerical features from the target labels.

The 30 numerical features will be used as input variables (X), while User_ID will be used as the target variable (y).

In [ ]:
# Separate Features and Target

X = voice_df.drop(columns=["User_ID", "Audio_Name"])
y = voice_df["User_ID"]

print("Features and Target Separated Successfully")
print("-" * 45)

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

# 04 - Split Training and Testing Data

This section splits the dataset into training and testing subsets.

The training set will be used to train the machine learning models, while the testing set will be used to evaluate their performance on unseen recordings.

Stratified sampling is applied to preserve the original class distribution in both subsets.

In [ ]:
# Split Dataset into Training and Testing Sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Dataset Split Completed Successfully")
print("-" * 45)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

# 05 - Feature Scaling

This section applies StandardScaler to normalize the numerical voice features.

Feature scaling is important for SVM because it is sensitive to differences in feature magnitudes.

The scaler is fitted only on the training data to prevent data leakage, then applied to both training and testing data

In [ ]:
# Apply Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature Scaling Completed Successfully")
print("-" * 45)

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

print("\nTraining Feature Mean (First 5):")
print(np.mean(X_train_scaled, axis=0)[:5])

print("\nTraining Feature Standard Deviation (First 5):")
print(np.std(X_train_scaled, axis=0)[:5])

# 06 - Random Forest Model Training

This section trains a Random Forest classifier using the combined MFCC, ZCR, and RMS features

In [ ]:
# Train Random Forest Model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully")
print("-" * 45)

print("Number of Estimators:", rf_model.n_estimators)
print("Training Samples:", X_train.shape[0])
print("Number of Features:", X_train.shape[1])

# 07 - SVM Model Training

This section trains a Support Vector Machine (SVM) classifier using the combined MFCC, ZCR, and RMS features.

In [ ]:
# Train SVM Model

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm_model.fit(X_train_scaled, y_train)

print("SVM Model Trained Successfully")
print("-" * 45)

print("Kernel:", svm_model.kernel)
print("C:", svm_model.C)
print("Gamma:", svm_model.gamma)
print("Training Samples:", X_train_scaled.shape[0])
print("Number of Features:", X_train_scaled.shape[1])

# 08 - Model Predictions

This section generates predictions using the trained Random Forest and SVM models.

In [ ]:
# Generate Model Predictions

rf_predictions = rf_model.predict(X_test)

svm_predictions = svm_model.predict(X_test_scaled)

print("Model Predictions Completed")
print("-" * 45)

print("Random Forest Predictions:")
print(rf_predictions)

print("\nSVM Predictions:")
print(svm_predictions)

print("\nActual Labels:")
print(y_test.to_numpy())

# 09 - Model Evaluation

This section evaluates the performance of the Random Forest and SVM models using standard classification metrics.

In [ ]:
# Evaluate Machine Learning Models

rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions, average="weighted")
rf_recall = recall_score(y_test, rf_predictions, average="weighted")
rf_f1 = f1_score(y_test, rf_predictions, average="weighted")

svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_precision = precision_score(y_test, svm_predictions, average="weighted")
svm_recall = recall_score(y_test, svm_predictions, average="weighted")
svm_f1 = f1_score(y_test, svm_predictions, average="weighted")

voice_results = pd.DataFrame({
    "Accuracy": [rf_accuracy, svm_accuracy],
    "Precision": [rf_precision, svm_precision],
    "Recall": [rf_recall, svm_recall],
    "F1-Score": [rf_f1, svm_f1]
}, index=["Random Forest", "SVM"])

print("Voice Model Evaluation Results")
print("-" * 45)

display(voice_results.round(4))

In [ ]:
# Generate Confusion Matrices

rf_cm = confusion_matrix(
    y_test,
    rf_predictions,
    labels=["User_01", "User_02"]
)

svm_cm = confusion_matrix(
    y_test,
    svm_predictions,
    labels=["User_01", "User_02"]
)

print("Random Forest Confusion Matrix:")
display(pd.DataFrame(
    rf_cm,
    index=["Actual User_01", "Actual User_02"],
    columns=["Predicted User_01", "Predicted User_02"]
))

print("\nSVM Confusion Matrix:")
display(pd.DataFrame(
    svm_cm,
    index=["Actual User_01", "Actual User_02"],
    columns=["Predicted User_01", "Predicted User_02"]
))

# 10 - Compare Voice Models

This section compares the performance of the Random Forest and SVM classifiers.

In [ ]:
# Compare Voice Machine Learning Models

print("Voice Models Comparison")
print("-" * 45)

display(voice_results.round(4))

print("\nBest Model Comparison:")

if voice_results["F1-Score"].idxmax() == "Random Forest":
    print("Random Forest achieved the highest F1-Score.")
elif voice_results["F1-Score"].idxmax() == "SVM":
    print("SVM achieved the highest F1-Score.")
else:
    print("Both models achieved the same F1-Score.")

# 11 - Voice Models Cross-Validation

This section evaluates the stability and generalization performance of the Random Forest and SVM classifiers using 5-Fold Stratified Cross-Validation.

The dataset is divided into five folds while preserving the class distribution.

Each model is trained on four folds and evaluated on the remaining fold. This process is repeated five times.

A Pipeline is used for SVM to ensure that feature scaling is performed independently within each training fold, preventing data leakage.

The mean Accuracy and F1-Score are calculated to compare the overall performance of both models.

In [ ]:
# Voice Models Cross-Validation

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Create SVM Pipeline
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

# Define Cross-Validation Metrics
scoring = {
    "accuracy": "accuracy",
    "f1": "f1_weighted"
}

# Random Forest Cross-Validation
rf_cv_results = cross_validate(
    rf_model,
    X,
    y,
    cv=cv,
    scoring=scoring
)

# SVM Cross-Validation
svm_cv_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

# Display Results
cv_results_df = pd.DataFrame({
    "Accuracy": [
        rf_cv_results["test_accuracy"].mean(),
        svm_cv_results["test_accuracy"].mean()
    ],
    "F1-Score": [
        rf_cv_results["test_f1"].mean(),
        svm_cv_results["test_f1"].mean()
    ]
}, index=["Random Forest", "SVM"])

print("Voice Models Cross-Validation Results")
print("-" * 50)

display(cv_results_df.round(4))

# 12 - Save Voice Models

This section saves the trained Random Forest and SVM models using Joblib.

The saved model bundles include:

- The trained classification model.
- The feature column names to preserve the original feature order.
- The StandardScaler for SVM.

These files will be used later during the voice verification and system integration stages.

In [ ]:
# Save Voice Machine Learning Models

MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Preserve Feature Column Order
feature_columns = X.columns.tolist()

# Random Forest Model Bundle
rf_bundle = {
    "model": rf_model,
    "feature_columns": feature_columns
}

# SVM Model Bundle
svm_bundle = {
    "model": svm_model,
    "scaler": scaler,
    "feature_columns": feature_columns
}

# Save Models
rf_model_file = MODEL_PATH / "Random_Forest_Voice_Recognition.joblib"
svm_model_file = MODEL_PATH / "SVM_Voice_Recognition.joblib"

joblib.dump(rf_bundle, rf_model_file)
joblib.dump(svm_bundle, svm_model_file)

print("Voice Models Saved Successfully")
print("-" * 45)

print(f"Random Forest: {rf_model_file}")
print(f"SVM: {svm_model_file}")

In [ ]:
# Load Saved Voice Models

loaded_rf_bundle = joblib.load(rf_model_file)
loaded_svm_bundle = joblib.load(svm_model_file)

# Extract Models and Scaler
loaded_rf_model = loaded_rf_bundle["model"]

loaded_svm_model = loaded_svm_bundle["model"]
loaded_svm_scaler = loaded_svm_bundle["scaler"]

# Extract Feature Columns
loaded_rf_columns = loaded_rf_bundle["feature_columns"]
loaded_svm_columns = loaded_svm_bundle["feature_columns"]

# Prepare Testing Features
X_test_rf = X_test[loaded_rf_columns]
X_test_svm = X_test[loaded_svm_columns]

X_test_svm_scaled = loaded_svm_scaler.transform(X_test_svm)

# Generate Predictions
loaded_rf_predictions = loaded_rf_model.predict(X_test_rf)
loaded_svm_predictions = loaded_svm_model.predict(X_test_svm_scaled)

# Verify Predictions
print("Saved Voice Models Test")
print("-" * 45)

print("Random Forest Predictions Match:",
      np.array_equal(rf_predictions, loaded_rf_predictions))

print("SVM Predictions Match:",
      np.array_equal(svm_predictions, loaded_svm_predictions))

print("\nActual Labels:")
print(y_test.to_numpy())

print("\nRandom Forest Loaded Predictions:")
print(loaded_rf_predictions)

print("\nSVM Loaded Predictions:")
print(loaded_svm_predictions)